[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/S07_03_regresion_logistica.ipynb)

# S07 · 03 — Regresión logística: predecir cancelaciones

**Tiempo estimado:** 70 min · **Sprint 7** · M3 (UD 3.4)

**Contexto TurisData.** El equipo de reservas quiere saber **qué reservas tienen riesgo de
cancelarse** para actuar a tiempo (pedir un depósito, hacer un recordatorio, vigilar el
*overbooking*). Es un problema de **clasificación** (0 = se mantiene, 1 = cancela).
La regresión logística es el punto de partida: rápida, explicable y da **probabilidades**.

**Al terminar sabrás:**
1. Explicar cómo una regresión logística convierte variables en una probabilidad.
2. Leer los coeficientes como *odds ratios* («esta característica multiplica el riesgo por…»).
3. Elegir un **umbral** de decisión y leer la **matriz de confusión**.
4. Escoger el umbral según el **coste** de cada tipo de error.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, accuracy_score

SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
df["temporada_alta"] = df["mes_llegada"].isin([1, 2, 3, 7, 8, 12]).astype(int)

columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
print(f"Tasa de cancelación: {y.mean():.1%} · train {len(X_train)} · test {len(X_test)}")

## 1. La idea: de una suma ponderada a una probabilidad

La regresión logística calcula, igual que la lineal, una **suma ponderada** `z` de las
variables. Pero en lugar de devolver `z` como respuesta, lo pasa por la **función
sigmoide**, que lo comprime entre 0 y 1: `probabilidad = 1 / (1 + e^(−z))`.
Así el resultado se puede leer como **la probabilidad de cancelar**.

In [ ]:
z = np.linspace(-6, 6, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(z, 1 / (1 + np.exp(-z)), color="#2a6f97", lw=2)
plt.axhline(0.5, color="gray", ls="--")
plt.title("Función sigmoide: convierte cualquier número en una probabilidad")
plt.xlabel("Suma ponderada z")
plt.ylabel("Probabilidad")
plt.show()

## 2. Entrenar el modelo

Como en el Sprint 6, todo el preprocesado va dentro de un `Pipeline` (imputar, escalar y
codificar). El modelo devuelve `predict_proba` (probabilidad por clase) y `predict`
(la clase, usando un **umbral de 0,5** por defecto).

**Ejercicio 1.** Crea `logit`, un `Pipeline` con el preprocesado que ves abajo (`prep`) y
`LogisticRegression(max_iter=1000)`. Ajústalo con train y guarda en `proba_test` la
probabilidad de cancelar (clase 1) de cada reserva de test.

In [ ]:
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))]), columnas_cat),
])
# TODO: Pipeline([("prep", prep), ("clf", LogisticRegression(...))]); predict_proba(X_test)[:, 1]
...
assert proba_test.shape == (len(X_test),), "Debe haber una probabilidad por reserva de test"
assert 0 <= proba_test.min() and proba_test.max() <= 1, "Son probabilidades: entre 0 y 1"
assert 0.35 < proba_test.mean() < 0.55, "La probabilidad media debería rondar la tasa de cancelación (≈0,46)"

## 3. Interpretar: coeficientes y *odds ratios*

Los *odds* son «probabilidad de cancelar / probabilidad de no cancelar». Al hacer
`exp(coeficiente)` obtenemos el ***odds ratio***: cuánto se **multiplican los odds** al subir
la variable una unidad (en las numéricas, una desviación típica por el escalado) o al
pasar a esa categoría.
- *Odds ratio* > 1 → **aumenta** el riesgo. < 1 → lo **reduce**. = 1 → no influye.

In [ ]:
nombres = logit[:-1].get_feature_names_out()
coefs = pd.Series(logit[-1].coef_[0], index=nombres)
odds = np.exp(coefs).sort_values(ascending=False)
plt.figure(figsize=(7, 6))
odds.iloc[::-1].plot(kind="barh", color=["#e63946" if v > 1 else "#2a6f97" for v in odds.iloc[::-1]])
plt.axvline(1, color="black", lw=1)
plt.title("Odds ratio de cada variable (rojo = más riesgo de cancelar)")
plt.xlabel("Odds ratio")
plt.ylabel("")
plt.tight_layout()
plt.show()

**Ejercicio 2.** A partir de `odds` (un `pd.Series` cuyo índice son los nombres de las
variables), guarda en `or_no_reembolsable` el odds ratio de `cat__tarifa_no_reembolsable`
y en `or_deposito_total` el de `cat__deposito_total`. Guarda en `mas_riesgo` el nombre de
la variable con **mayor** odds ratio.

In [ ]:
# TODO: odds["nombre_de_la_variable"]; idxmax() para el mayor
...
assert or_no_reembolsable < 1, "La tarifa no reembolsable reduce el riesgo de cancelar (odds ratio < 1)"
assert or_deposito_total < 1, "Un depósito total reduce el riesgo de cancelar (odds ratio < 1)"
assert isinstance(mas_riesgo, str) and odds[mas_riesgo] > 1
print(f"Tarifa no reembolsable: los odds de cancelar se multiplican por {or_no_reembolsable:.2f}")
print(f"Depósito total: los odds de cancelar se multiplican por {or_deposito_total:.2f}")

## 4. El umbral y la matriz de confusión

El modelo da una probabilidad; **nosotros decidimos** a partir de qué valor lo llamamos
«riesgo de cancelación». Esa decisión es el **umbral**.

La **matriz de confusión** cuenta los cuatro resultados posibles (con «cancela» = positivo):

| | Predicho: se mantiene | Predicho: cancela |
|---|---|---|
| **Real: se mantiene** | Verdadero negativo (VN) | **Falsa alarma** (FP) |
| **Real: cancela** | **Cancelación no prevista** (FN) | Verdadero positivo (VP) |

- **Precisión** = VP / (VP + FP): de las alarmas, ¿cuántas eran ciertas?
- **Recall** = VP / (VP + FN): de las cancelaciones reales, ¿cuántas avisamos?

In [ ]:
def metricas_umbral(y_real, proba, umbral):
    """Matriz de confusión y métricas para un umbral dado."""
    pred = (proba >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_real, pred).ravel()
    return {"umbral": umbral, "VN": vn, "FP": fp, "FN": fn, "VP": vp,
            "precision": vp / (vp + fp) if (vp + fp) else np.nan,
            "recall": vp / (vp + fn)}

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for ax, u in zip(ejes, [0.5, 0.3]):
    ConfusionMatrixDisplay.from_predictions(y_test, (proba_test >= u).astype(int), ax=ax,
                                            display_labels=["Se mantiene", "Cancela"], cmap="Blues", colorbar=False)
    ax.set_title(f"Umbral {u}")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

**Ejercicio 3.** Usando `metricas_umbral`, calcula sobre test (`y_test`, `proba_test`) las
métricas con umbral 0,5 (`m05`) y con umbral 0,3 (`m03`). Guarda en `sube_recall` la
diferencia de recall (`m03` − `m05`) y en `baja_precision` la de precisión (`m05` − `m03`).

In [ ]:
# TODO: llama a metricas_umbral dos veces (umbral 0.5 y 0.3) y resta los campos que necesites
...
assert m05["VN"] + m05["FP"] + m05["FN"] + m05["VP"] == len(y_test), "La matriz debe sumar todas las reservas de test"
assert sube_recall > 0, "Bajar el umbral debe subir el recall"
assert baja_precision > 0, "Bajar el umbral suele bajar la precisión"
print(f"Con umbral 0,3 detectas {sube_recall:.0%} más de las cancelaciones, pero tu precisión baja {baja_precision:.0%}.")

## 5. Elegir el umbral según el coste

**Idea.** Ambos errores no cuestan lo mismo:
- **Cancelación no prevista (FN):** habitación vacía que podíamos haber vendido.
- **Falsa alarma (FP):** molestamos a un cliente que iba a venir, o hacemos *overbooking* sin necesidad.

Si un FN cuesta el **triple** que un FP, conviene ser más "alarmista" (umbral bajo). La regla
teórica para el umbral óptimo es `coste_FP / (coste_FP + coste_FN)`; aquí lo comprobamos
**empíricamente**. El umbral se elige con probabilidades **de validación cruzada sobre
train** (nunca con el test, que se reserva para la evaluación final).

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)
proba_cv = cross_val_predict(logit, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

def coste_total(y_real, proba, umbral, coste_fn=3.0, coste_fp=1.0):
    m = metricas_umbral(y_real, proba, umbral)
    return m["FN"] * coste_fn + m["FP"] * coste_fp

**Ejercicio 4.** Prueba los umbrales `umbrales = np.arange(0.05, 0.96, 0.05)` sobre
`proba_cv` / `y_train` con `coste_total` (FN = 3, FP = 1) y guarda en `mejor_umbral` el que
da **menor coste**. Después calcula en `coste_test` el coste de ese umbral sobre test.

In [ ]:
umbrales = np.arange(0.05, 0.96, 0.05)
# TODO: lista de costes con un for; np.argmin te da la posición del menor
...
assert 0.1 <= mejor_umbral <= 0.4, "Con FN 3 veces más caro que FP, el umbral óptimo debería ser bajo (≈0,25)"
assert coste_test < coste_total(y_test, proba_test, 0.5), "El umbral elegido debería costar menos que el 0,5 por defecto"
print(f"Umbral elegido: {mejor_umbral:.2f} · coste en test: {coste_test:.0f} (con 0,5: {coste_total(y_test, proba_test, 0.5):.0f})")

In [ ]:
plt.figure(figsize=(7, 3.8))
plt.plot(umbrales, costes, marker="o", color="#2a6f97")
plt.axvline(mejor_umbral, color="#e63946", ls="--", label=f"Mínimo: {mejor_umbral:.2f}")
plt.title("Coste total según el umbral (FN cuesta 3 veces más que FP)")
plt.xlabel("Umbral de decisión")
plt.ylabel("Coste (unidades relativas)")
plt.legend()
plt.show()

## 6. Mini-reto: tabla para el gerente

Construye `tabla_umbrales`, un DataFrame con **tres filas** (umbrales 0,5, 0,3 y el
`mejor_umbral`) y las columnas `umbral`, `recall`, `precision` y `coste` medidas en **test**.
Después escribe en `recomendacion` (texto) qué umbral propondrías y por qué, con al menos una cifra.

In [ ]:
# TODO: una lista de umbrales, metricas_umbral y coste_total en cada uno
...
assert tabla_umbrales.shape[0] == 3, "Debe haber tres filas"
assert {"umbral", "recall", "precision", "coste"} <= set(tabla_umbrales.columns)
assert isinstance(recomendacion, str) and any(c.isdigit() for c in recomendacion), "La recomendación debe ser texto con al menos una cifra"
tabla_umbrales

## 7. Para reflexionar

1. Un modelo con exactitud 0,67 ¿es bueno para TurisData? ¿Qué preguntas harías antes de contestar?
2. El *odds ratio* de la tarifa no reembolsable es < 1. ¿Significa que **causa** menos cancelaciones? ¿Qué otra explicación hay?
3. ¿Qué pasa con la precisión y el recall si subes el umbral a 0,8?
4. ¿Quién debería fijar los costes de FN y FP: el equipo de datos o el gerente? ¿Por qué?

## 8. Resumen: qué has aprendido
- La regresión logística devuelve **probabilidades**; el umbral las convierte en decisiones.
- `exp(coeficiente)` = *odds ratio*: explica el efecto de cada variable en el riesgo.
- La matriz de confusión separa dos errores con costes distintos.
- El umbral se elige según el negocio y con validación, no con el test.